# Session 6: Environmental Justice & Vector Data

**Goal:** Analyze the environmental justice implications of data center siting using real demographic and historical data.

Datasets:
1.  **Data Centers**: Locations from Cloud Regions.
2.  **Social Vulnerability Index (SVI)**: CDC's ranking of community vulnerability (Census Tracts).
3.  **Redlining (HOLC)**: Historical maps grading neighborhoods for mortgage risk.

Key concepts:
1.  **Spatial Joins**: Linking indices to infrastructure.
2.  **Historical Context**: Overlapping past policies with current infrastructure.

In [1]:
import ibis
import pandas
from ibis import _
from anymap import Map
import geopandas as gpd
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import geopandas as gpd
from shapely import wkb

ibis.options.interactive = True
con = ibis.duckdb.connect()
con.raw_sql("INSTALL spatial; LOAD spatial;")

## 1. Load Data

We load the Data Centers, SVI data (Parquet), and Redlining data (Parquet).

In [2]:
# 1. Data Centers
dc_url = "https://s3-west.nrp-nautilus.io/public-datacenters/data_centers.csv"
# Load via connection to ensure same backend as SVI
dcs = con.read_csv(dc_url).filter(_.country == 'United States')
# Create point geometry using the longitude.point(latitude) method
# Cast latitude to float as it might be inferred as string
dcs_geo = dcs.mutate(geom = dcs.longitude.point(dcs.latitude.cast('float')))

# 2. Social Vulnerability Index (SVI) 2020 - Census Tracts
svi_url = "https://s3-west.nrp-nautilus.io/public-social-vulnerability/svi2020_us_tract.parquet"
# Load via SQL to ensure proper schema inference with spatial extension
svi = con.sql(f"SELECT * FROM read_parquet('{svi_url}')").rename(geometry="Shape")

# 3. Redlining (Mapping Inequality) - Polygons
red_url = "https://s3-west.nrp-nautilus.io/public-mappinginequality/mappinginequality.parquet"
# Load via SQL to ensure proper schema inference with spatial extension
redlining = con.sql(f"SELECT * FROM read_parquet('{red_url}')")

print("Datums loaded.")

Datums loaded.


## 2. Visualize SVI Layers

The SVI dataset contains a ranking `RPL_THEMES` (0 to 1), where higher values indicate higher vulnerability.

In [3]:
# Convert SVI data to GeoDataFrame
svi_df = svi.select("RPL_THEMES", "geometry").to_pandas()
svi_gdf = gpd.GeoDataFrame(svi_df, geometry='geometry')

# Convert data centers to GeoDataFrame
dcs_gdf = dcs_geo.to_pandas()
dcs_gdf = gpd.GeoDataFrame(dcs_gdf, geometry='geom', crs='EPSG:4326')

# Create colormap for RPL_THEMES values
norm = mcolors.Normalize(vmin=svi_gdf['RPL_THEMES'].min(), vmax=svi_gdf['RPL_THEMES'].max())
cmap = cm.viridis

# Create a Map
m = Map(center=[-98, 39], zoom=4, height="700px")

# Add SVI polygons with color based on RPL_THEMES
# Create a simplified GeoDataFrame with colors
svi_styled = svi_gdf.copy()
svi_styled['color'] = svi_styled['RPL_THEMES'].apply(
    lambda x: mcolors.to_hex(cmap(norm(x))) if pandas.notna(x) else '#cccccc'
)

m.add_vector(
    svi_styled,
    layer_type="fill",
    paint={
        "fill-color": ["get", "color"],
        "fill-opacity": 0.6
    },
    name="svi"
)

# Add data centers as points
m.add_vector(
    dcs_gdf,
    layer_type="circle",
    paint={
        "circle-radius": 4,
        "circle-color": "#ff0000"
    },
    name="data_centers"
)

m

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/home/cboettig/Documents/github/csol-208/module-2-template/.venv/lib/python3.12/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


## 3. Spatial Join: Vulnerability at Data Center Locations

We want to find the SVI score of the tract where each data center is located.

In [4]:
# Spatial Join: Points IN Polygons
# We renamed SVI geometry to 'geometry'
joined = (
    dcs_geo
    .join(svi, dcs_geo.geom.within(svi.geometry))
    .select("provider", "metro", "RPL_THEMES", "FIPS")
)

joined.head()

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ provider            ┃ metro   ┃ RPL_THEMES ┃ FIPS        ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ string              │ string  │ float64    │ string      │
├─────────────────────┼─────────┼────────────┼─────────────┤
│ Amazon Web Services │ Phoenix │     0.4322 │ 04013114100 │
│ OVH                 │ Phoenix │     0.4322 │ 04013114100 │
│ Microsoft Azure     │ Phoenix │     0.4322 │ 04013114100 │
│ Microsoft Azure     │ Phoenix │     0.4322 │ 04013114100 │
│ Google Cloud        │ Phoenix │     0.4322 │ 04013114100 │
└─────────────────────┴─────────┴────────────┴─────────────┘

### Analyze Findings
Are data centers located in highly vulnerable areas (`RPL_THEMES` > 0.75)?

In [5]:
joined.group_by("provider").aggregate(avg_svi = _.RPL_THEMES.mean()).order_by(_.avg_svi.desc())

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ provider            ┃ avg_svi    ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ string              │ float64    │
├─────────────────────┼────────────┤
│ Alibaba Cloud       │   0.521482 │
│ Tencent Cloud       │   0.481722 │
│ OVH                 │   0.461995 │
│ Microsoft Azure     │ -38.604904 │
│ Amazon Web Services │ -40.544626 │
│ IBM Cloud           │ -42.863300 │
│ Google Cloud        │ -63.520194 │
│ Oracle Cloud        │ -70.981029 │
└─────────────────────┴────────────┘

## 4. Redlining Overlay

The HOLC redlining maps graded neighborhoods from 'A' (Best) to 'D' (Hazardous - Red).
Do any data centers fall within historically redlined districts?

In [6]:
# Filter for only grade 'D' (Redlined) areas
redlined_areas = redlining.filter(redlining.grade == 'D')

dc_in_redlined = (
    dcs_geo
    # redlining has geometry in 'geom' column
    .join(redlined_areas, dcs_geo.geom.within(redlined_areas.geom))
    .select("provider", "metro", "grade", "city")
)

dc_in_redlined

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ provider            ┃ metro      ┃ grade  ┃ city       ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ string              │ string     │ string │ string     │
├─────────────────────┼────────────┼────────┼────────────┤
│ Google Cloud        │ Columbus   │ D      │ Columbus   │
│ Microsoft Azure     │ Des Moines │ D      │ Des Moines │
│ Microsoft Azure     │ Des Moines │ D      │ Des Moines │
│ Microsoft Azure     │ Des Moines │ D      │ Des Moines │
│ Amazon Web Services │ Chicago    │ D      │ Chicago    │
│ Amazon Web Services │ Chicago    │ D      │ Chicago    │
│ Google Cloud        │ Chicago    │ D      │ Chicago    │
│ Amazon Web Services │ Chicago    │ D      │ Chicago    │
│ Amazon Web Services │ Chicago    │ D      │ Chicago    │
│ Microsoft Azure     │ Chicago    │ D      │ Chicago    │
│ …                   │ …          │ …      │ …          │
└─────────────────────┴────────────┴────────┴────────────┘

## Discussion

How does the geography of the cloud align with the geography of inequality? (Note: Many data centers are in newer suburbs (e.g. Ashburn) that post-date redlining maps, so overlaps might be limited to urban interconnection hubs).